# Suite DST — Distributions

A pattern is a predicate: `Distributions.Choices` weighs alternatives, its arms, and holds when the number of its arms
that hold satisfies its count; a distribution of values binds a symbol, within its body, to a value drawn from it, and
holds of data when the value its body equates the symbol with is in its support. `Generators.Sample` draws a store's
matches by what they weigh.

In [ ]:
import { Expressions as E } from "@mbse/expressions";
import { Terms } from "@mbse/expressions/Framework";
import { Constraints as C, Distributions as D, Generators as G, Predicates as P, register } from "@mbse/patterns";
import { Errors, JSON as SchemaJSON, Proxies, Schemas as S, Stores } from "@mbse/schemas/Framework";
import { Contact, Phone, assert, book, listed, raises, same } from "./support.js";

const APerson = { person: Contact };
const [person, age, x] = [E.variable("person"), E.variable("age"), E.variable("x")];
const HasName = new P.OfPredicate.Builder().name("HasName").symbols(APerson).parameters((p) => p.name("name"))
  .requires(person.name.eq(E.variable("name"))).create();
const Names = new P.OfPredicate.Builder().name("Names").symbols(APerson).requires(new D.Choices.Builder().arms(
  (a) => a.weight(10).requires(HasName.call(person, "alice")),
  (a) => a.weight(5).requires(HasName.call(person, "ben")),
  (a) => a.weight(15).requires(HasName.call(person, "chermon")),
  (a) => a.weight(7).requires(HasName.call(person, "davi")),
).decreasing().create()).create();

function counted(names: string[]): Map<string, number> {
  const counts = new Map<string, number>();
  for (const name of names) counts.set(name, (counts.get(name) ?? 0) + 1);
  return counts;
}

function take<T>(items: Iterator<T>, n: number): T[] {
  return Array.from({ length: n }, () => items.next().value as T);
}

## DST-01 · Choices are built fluently: arms, a count and a precedence

In [ ]:
{
  const choices = Names.rule as D.Choices;
  assert(choices instanceof D.Choices && choices.decreasing === true && same(choices.arms.map((a) => a.weight), [10, 5, 15, 7]));
  assert(choices.arms.every((arm) => arm.condition.predicate === HasName)); // one predicate, by reference
  assert(choices.count.name === "c" && same(choices.count.binds(), ["c"]) && new D.Count().binds().length === 0);
  assert(Terms.same(choices.count.condition, E.variable("c").ge(1n).data)); // at least one arm, by default
  const both = new D.Choices.Builder().arms((a) => a.weight(1).requires(person.has("name"), person.has("age"))).count(
    (n) => n.eq(1n)).create(); // an arm's conditions are conjoined; the count is what the function writes
  assert(Terms.same(both.arms[0].condition, person.has("name").and_(person.has("age")).data) && both.decreasing === null);
  assert(both.count.name === "n" && Terms.same(both.count.condition, E.variable("n").eq(1n).data));
  const grown = new D.Choices.Builder(both).arms((a) => a.weight(2).requires(true)).clone();
  assert(grown.arms.length === 2 && both.arms.length === 1 && grown.count === both.count); // the count is kept
  assert(new D.Choices.Builder(grown).decreasing().update() === grown && grown.decreasing === true);
  const arm = D.Arm.resolve((a: D.Arm.Builder) => a.weight(2).requires(true).requires(false));
  assert(arm.weight === 2 && Terms.same(arm.condition, E.literal(true).and_(false).data) && D.Arm.resolve(arm) === arm);
  assert(Terms.same(new D.Arm.Builder(arm).requires(x).clone().condition, E.operation("and", arm.condition, x).data));
  assert(new D.Arm.Builder(arm).weight(3).update() === arm && arm.weight === 3);
  assert(same(P.DIALECT.validate(new D.Arm.Builder().weight(1).create()), ["an arm needs a condition"]));
  raises(TypeError, () => D.Arm.resolve("x"), "expected an arm or a callable taking its builder, got 'x'");
  raises(TypeError, () => new D.Choices.Builder().count(((a: any, b: any) => true) as any), "a choices' count is a function of one number");
  assert(P.DIALECT.validate(Names, { core: true }).length === 0);
  assert(same(P.DIALECT.validate(new D.Choices.Builder().create()), ["a choices needs an arm"]));
  assert(same(P.DIALECT.validate(new D.Choices(null, both.count, [E.literal(1n).data])), ["a choices' arguments after its count are arms"]));
  assert(same(P.DIALECT.validate(new D.Choices(null, E.literal(1n).data, both.arms), { bound: ["person"] }), ["a choices' count must be a count"]));
  assert(same(P.DIALECT.validate(new D.Choices.Builder().arms((a) => a.weight(-1).requires(true)).create()), [
    "argument 0: an arm's weight must be positive, got -1.0"]));
}

## DST-02 · Choices hold when their count does; a match weighs what its arms say

In [ ]:
{
  const store = book();
  const [alice, ben, zed] = ["alice", "ben", "zed"].map((n) => store.Contact().name(n).create());
  const evaluate = new P.Evaluator(store);
  assert(same([alice, ben, zed].map((p) => P.holds(evaluate, Names.rule, { person: p })), [true, true, false]));
  const Two = new D.Choices.Builder().arms((a) => a.weight(1).requires(person.has("name")),
    (a) => a.weight(2).requires(HasName.call(person, "alice"))).count((c) => c.eq(2n)).create();
  assert(same([alice, ben].map((p) => evaluate.run(Two, { person: p })), [true, false])); // both arms hold of alice only
  const unknown = new D.Choices.Builder().arms((a) => a.weight(1).requires(x), (a) => a.weight(1).requires(true)).create();
  assert(evaluate.run(unknown, { x: null }) === true); // one arm holds: at least one, whatever the other
  assert(evaluate.run(new D.Choices.Builder(unknown).count((c) => c.eq(1n)).clone(), { x: null }) === null); // 1 or 2
  raises(TypeError, () => evaluate.run(new D.Choices.Builder().arms((a) => a.weight(1).requires(3n)).create()),
    "a predicate must be a bool, got int");
  assert(same([alice, ben, zed].map((p) => evaluate.weigh(Names.rule, { person: p })), [10, 5, 0]));
  const overlapping = () => new D.Choices.Builder().arms((a) => a.weight(3).requires(person.has("name")),
    (a) => a.weight(2).requires(HasName.call(person, "alice")));
  assert(evaluate.weigh(overlapping().decreasing().create(), { person: alice }) === 3); // the first that holds
  assert(evaluate.weigh(overlapping().create(), { person: alice }) === 5); // without precedence, every arm that holds
  const none = new D.Choices.Builder().arms((a) => a.weight(1).requires(person.has("pager"))).count((c) => c.eq(0n)).decreasing().create();
  assert(evaluate.run(none, { person: alice }) === true && evaluate.weigh(none, { person: alice }) === 0); // no arm to fall under
  const Nested = new P.OfPredicate.Builder().symbols(APerson).requires(person.has("name")).requires(Names.rule).requires(
    new D.Choices.Builder().arms((a) => a.weight(2).requires(new D.Choices.Builder().arms(
      (b) => b.weight(4).requires(true)).create())).create()).create();
  assert(evaluate.weigh(Nested.rule, { person: alice }) === 10 * 2 * 4); // conjuncts multiply, arms nest
  const applied = new P.OfPredicate.Builder().symbols({ who: Contact }).requires(Names.call(E.variable("who"))).create();
  assert(evaluate.weigh(applied.rule, { who: ben }) === 5); // through an application
}

## DST-03 · Distributions of values bind a symbol, and are built fluently

In [ ]:
{
  const normal = new D.Normal.Builder().symbol("age").mean(70n).deviation(8n).rounded().requires(
    (age: any) => person.age.eq(age)).create();
  assert(normal.symbol === "age" && normal.rounded === true && normal.mean.value === 70n && normal.deviation.value === 8n);
  assert(same(normal.binds(), ["age"]) && Terms.same(normal.body, person.age.eq(age).data)); // what the function writes
  assert(new D.Normal().binds().length === 0); // without a symbol, nothing
  const builder = new D.Uniform.Builder().symbol("n").low(1n).high(6n);
  assert(Terms.same(builder.n.data, E.variable("n").data)); // the builder gives its symbol as a variable
  assert(builder.m === undefined); // Python raises AttributeError
  const uniform = builder.requires(person.age.eq(builder.n), person.has("name")).create(); // specs, conjoined
  assert(Terms.same(uniform.body, person.age.eq(E.variable("n")).and_(person.has("name")).data));
  assert(Terms.same(new D.Uniform.Builder(uniform).requires(true).clone().body, E.operation("and", uniform.body, true).data));
  assert(new D.Uniform.Builder(uniform).high(9n).update() === uniform && uniform.high.value === 9n);
  const categorical = new D.Categorical.Builder().symbol("name").option(3, "ann").option(1, "bo").requires(
    person.name.eq(E.variable("name"))).create();
  assert(JSON.stringify(categorical.options.map((o) => [o.weight, o.value.value])) === JSON.stringify([[3, "ann"], [1, "bo"]]));
  const poisson = new D.Poisson.Builder().symbol("k").rate(1.5).create();
  const geometric = new D.Geometric.Builder().symbol("k").probability(0.5).create();
  assert(same([normal, new D.Normal.Builder().mean(1.0).deviation(1.0).create(), poisson, geometric, uniform,
    new D.Uniform.Builder().low(1n).high(2.5).create(), new D.Uniform.Builder().low(x).high(2n).create(), categorical,
    new D.Categorical.Builder().option(1, "a").option(2, 3n).create(), E.literal(true).data, x.data].map((d) => D.domain(d)),
  ["int", "float", "int", "int", "int", "float", null, "str", null, "bool", null]));
  assert(P.DIALECT.validate(normal, { bound: ["person"] }).length === 0);
  assert(same(P.DIALECT.validate(new D.Normal.Builder().mean(1n).deviation(1n).create()), ["a normal needs a symbol", "a normal needs a body"]));
  assert(same(P.DIALECT.validate(new D.Categorical.Builder().symbol("v").requires(true).create()), ["a categorical needs an option"]));
  assert(same(P.DIALECT.validate(new D.Categorical("v", E.literal(true).data, [E.literal(1n).data])), [
    "a categorical's arguments after its body are options"]));
  assert(same(P.DIALECT.validate(new D.Categorical.Builder().symbol("v").option(-1, 1n).requires(true).create()), [
    "argument 0: an option's weight must be positive, got -1.0"]));
  assert(Terms.same(D.witness(normal), person.age.data)); // what the body equates the symbol with
  const looping = new D.Poisson.Builder().symbol("k").requires(E.variable("k").eq(E.variable("k").add(1n))).create();
  assert(D.witness(looping) === null); // a witness does not read the symbol itself
  raises(TypeError, () => new D.Poisson.Builder().requires((k: any) => k), "a distribution's symbol is set before a function writes its conditions");
}

## DST-04 · A distribution holds of data when its witness is in its support

In [ ]:
{
  const store = book();
  const people = [30n, -2n, null].map((a) => a !== null ? store.Contact().age(a).create() : store.Contact().create());
  const evaluate = new P.Evaluator(store);
  const held = (make: () => any) => {
    const rule = make().symbol("v").requires(person.age.eq(E.variable("v"))).create();
    return people.map((p) => P.holds(evaluate, rule, { person: p }));
  };
  assert(same(held(() => new D.Normal.Builder().mean(0n).deviation(1n).rounded()), [true, true, null])); // any int; no witness
  assert(same(held(() => new D.Normal.Builder().mean(0n).deviation(1n)), [true, true, null])); // any number: 30 is 30.0
  assert(same(held(() => new D.Uniform.Builder().low(18n).high(64n)), [true, false, null]));
  assert(same(held(() => new D.Poisson.Builder().rate(2n)), [true, false, null])
    && same(held(() => new D.Geometric.Builder().probability(0.5)), [true, false, null]));
  assert(same(held(() => new D.Categorical.Builder().option(1, 30n).option(1, x)), [true, false, null]));
  const floats = new D.Uniform.Builder().symbol("v").low(0.0).high(1.0).requires(E.variable("v").eq(x)).create();
  assert(same([0.0, 0.5, 1.0, "a"].map((v) => evaluate.run(floats, { x: v })), [true, true, false, false])); // [low, high)
  const extra = new D.Uniform.Builder().symbol("v").low(0n).high(100n).requires(person.age.eq(E.variable("v")), E.variable("v").gt(50n)).create();
  assert(evaluate.run(extra, { person: people[0] }) === false); // in the support, but the body does not hold
  assert(evaluate.run(new D.Normal.Builder().symbol("v").mean(0n).deviation(1n).requires(true).create()) === null); // no witness
  const Wrong = new P.OfPredicate.Builder().name("Wrong").symbols(APerson).requires(
    new D.Normal.Builder().symbol("v").mean(0n).deviation(1n).requires(person.age.eq(E.variable("v"))).create(),
  ).requires(new D.Categorical.Builder().symbol("w").option(1, 2n).requires(E.variable("w").eq(person.name)).create(),
  ).requires(new D.Normal.Builder().symbol("u").mean(0n).deviation(1n).requires(x.age.eq(E.variable("u")), person.age.eq(1n)).create(),
  ).create();
  assert(same(D.typing(Wrong), ["person.age is int, but its distribution gives float", "person.name is str, but its distribution gives int"]));
  raises(Errors.ValueError, () => C.check([Wrong]), "predicate 'Wrong': person.age is int, but its distribution gives float");
  const Tagged = new S.OfObject.Builder().name("Tagged").ref().properties(
    (p) => p.name("tags").of((t) => t.as_indexed((i) => i.of((u) => u.as_native(String))))).create();
  const Tags = new P.OfPredicate.Builder().symbols({ it: Tagged }).requires(new D.Poisson.Builder().symbol("v").rate(1n).requires(
    E.variable("it").tags.eq(E.variable("v"))).create()).create();
  assert(D.typing(Tags).length === 0); // a property that is not a native: its type is not checked
  const Fine = new P.OfPredicate.Builder().symbols(APerson).requires(new D.Uniform.Builder().symbol("v").low(1n).high(6n).requires(
    person.age.eq(E.variable("v")), E.variable("v").eq(3n)).create()).create();
  assert(D.typing(Fine).length === 0); // an int, for an int; an equality that sets no property is passed over
  const outer = new P.OfPredicate.Builder().symbols(APerson).requires(Wrong.call(person)).create();
  assert(D.typing(outer).length === 0); // an applied predicate's distributions are its own to check
  assert(D.typing(new P.OfPredicate.Builder().create()).length === 0); // no rule, nothing to check
}

## DST-05 · Sample draws the store's matches in proportion to what they weigh

In [ ]:
{
  const store = book();
  listed(store, ...["alice", "ben", "chermon", "davi", "zed"].map((n) => store.Contact().name(n).create()));
  const drawn = G.Sample(store, Names, new Stores.PCG32(42n));
  const names = counted(take(drawn, 3700).map((m: any) => m.person.name));
  assert(same([...names.keys()].sort(), ["alice", "ben", "chermon", "davi"])); // zed weighs nothing
  for (const [n, w] of [["alice", 10], ["ben", 5], ["chermon", 15], ["davi", 7]] as const) {
    assert(Math.abs((names.get(n) as number) - 100 * w) < 100, n);
  }
  const again = [0, 1, 2].map(() => (G.Sample(store, Names, new Stores.PCG32(42n)).next().value as any).person.name);
  assert(same(again, [again[0], again[0], again[0]])); // the same seed, the same draws
  const first = take(G.Sample(store, Names, new Stores.PCG32(42n)), 6).map((m: any) => m.person.name);
  assert(!same(first, take(G.Sample(store, Names, new Stores.PCG32(42n).split("other")), 6).map((m: any) => m.person.name)));
  const nobody = book();
  listed(nobody, nobody.Contact().name("zed").create());
  raises(Errors.ValueError, () => G.Sample(nobody, Names, new Stores.PCG32(1n)).next(), "no match of the store's data has a weight");
  raises(Errors.ValueError, () => G.Sample(store, HasName, new Stores.PCG32(1n)),
    "the predicate cannot be drawn from: a predicate with parameters is drawn from where it is applied");
}

## DST-06 · Patterns are data: a predicate used by several arms is written once

In [ ]:
{
  const store = new C.OfStore(book());
  let text = SchemaJSON.ToJSON(store).Reachable(P.OfPredicate.Schema, Names);
  assert(text.split('"name": "HasName"').length - 1 === 1 && text.includes('"$schema": "Patterns.OfChoices"') && text.includes('"decreasing": true'));
  const copy = SchemaJSON.FromJSON(store).Reachable(P.OfPredicate.Schema, text) as P.OfPredicate;
  assert(Terms.same(copy, Names) && new Set(copy.rule.arms.map((arm: any) => arm.condition.predicate)).size === 1);
  const People = new P.OfPredicate.Builder().symbols(APerson).requires(new D.Choices.Builder().arms((a) => a.weight(1).requires(
    new D.Normal.Builder().symbol("age").mean(70n).deviation(8n).rounded().requires((age: any) => person.age.eq(age)).create(),
    new D.Categorical.Builder().symbol("name").option(3, "ann").option(1, "bo").requires(person.name.eq(E.variable("name"))).create(),
  )).create()).create();
  text = SchemaJSON.ToJSON(store).Reachable(P.OfPredicate.Schema, People);
  assert(text.includes('"$schema": "Patterns.OfNormal"') && text.includes('"rounded": true') && text.includes('"$schema": "Patterns.OfOption"'));
  assert(Terms.same(SchemaJSON.FromJSON(store).Reachable(P.OfPredicate.Schema, text), People));
}

## DST-07 · Patterns meet quantifiers, sets and stores

In [ ]:
{
  const store = book();
  listed(store, ...["ann", "bob"].map((n) => store.Contact().name(n).create()));
  const ann = store.Contact().name("ann").phones((e: any) => e.phone((q: any) => q.number("1"))).create();
  listed(store, ann);
  const p = E.variable("p");
  const owns = P.Exists((q) => q.symbols({ p: Phone }).requires(P.Contains(person.phones, (e) => e.phone.eq(p))));
  const Owning = new P.OfPredicate.Builder().name("Owning").symbols(APerson).requires(new D.Choices.Builder().arms(
    (a) => a.weight(2).requires(owns), (a) => a.weight(1).requires(person.has("name"))).decreasing().create()).create();
  const evaluate = new P.Evaluator(store);
  assert(same([...store.extent("Contact")].map((who) => evaluate.weigh(Owning.rule, { person: who })), [1, 1, 2])); // a link weighs
  const predicates = C.check([Names, Owning]); // patterns in a set, checked as predicates are
  assert(same(predicates.predicates.map((q: any) => q.name), ["Names", "Owning"]));
  const schemas = new C.OfStore(book());
  const copy = SchemaJSON.FromJSON(schemas).Reachable(C.OfSet.Schema, SchemaJSON.ToJSON(schemas).Reachable(C.OfSet.Schema, predicates)) as P.OfSet;
  assert(copy.predicates.every((a, i) => Terms.same(a, predicates.predicates[i])));
  const proxies = register(new Proxies.OfStore());
  for (const name of ["Patterns.OfChoices", "Patterns.OfArm", "Patterns.OfCount", "Patterns.OfNormal", "Patterns.OfOption"]) {
    assert(proxies.names().includes(name), name);
  }
  const Not = new P.OfPredicate.Builder().symbols(APerson).forbids(Names.rule).create(); // a pattern's negation is a predicate
  assert(same([...store.extent("Contact")].map((who) => P.holds(evaluate, Not.rule, { person: who })), [true, true, true]));
  assert(evaluate.weigh(Not.rule, { person: ann }) === 1); // choices under not weigh nothing
}